# Analyse prospects — notebook paramétrable par secteur

1. Choisir **`SECTOR_ID`** ci-dessous (clé de `scripts/prospecting/sectors.json`).
2. Exécuter toutes les cellules.

Prérequis : `pip install pandas matplotlib` — et avoir généré le CSV (`fetch_sector_prospects.py --sector <id>`).

In [ ]:
import json
from pathlib import Path

# --- À modifier : identifiant secteur (voir fetch_sector_prospects.py --list) ---
SECTOR_ID = "creches"

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

REG = json.loads((ROOT / "scripts" / "prospecting" / "sectors.json").read_text(encoding="utf-8"))
if SECTOR_ID not in REG["sectors"]:
    raise ValueError(f"Secteur inconnu: {SECTOR_ID!r}")
CFG = REG["sectors"][SECTOR_ID]
CSV_PATH = ROOT / "data" / CFG["csv_filename"]
print("Secteur :", CFG["label"])
print("NAF :", ", ".join(CFG["naf_codes"]))
print("CSV :", CSV_PATH)
assert CSV_PATH.is_file(), f"Export manquant — lancer : python scripts/prospecting/fetch_sector_prospects.py --sector {SECTOR_ID}"

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 55)

df = pd.read_csv(
    CSV_PATH,
    dtype={"siren": str, "siege_code_postal": str, "siege_departement": str},
)
df.head(8)

## Vue d’ensemble

In [ ]:
print(f"Lignes : {len(df):,}  |  SIREN uniques : {df['siren'].nunique():,}")
print(f"Doublons SIREN : {df['siren'].duplicated().sum()}")
missing = df.isna().sum().sort_values(ascending=False)
display(pd.DataFrame({"manquants": missing, "%": (100 * missing / len(df)).round(1)}).query("manquants > 0"))

## Tranches d’effectif, catégories, départements, établissements

Même logique que les notebooks historiques par secteur ; régénérer les PNG pour la doc :
`python scripts/prospecting/export_analysis_figures.py --sector <SECTOR_ID>`

In [ ]:
TRANCHE_LABELS = {
    "NN": "Non employeuse / NR", "00": "0", "01": "1-2", "02": "3-5", "03": "6-9",
    "11": "10-19", "12": "20-49", "22": "50-99", "31": "100-199", "32": "200-249",
    "41": "250-499", "42": "500-999", "51": "1k-2k", "52": "2k-5k", "53": "5k+",
}
vc_t = df["tranche_effectif_salarie"].value_counts().sort_index()
tbl = vc_t.rename("nombre").reset_index()
tbl.columns = ["code", "nombre"]
tbl["libelle"] = tbl["code"].astype(str).map(TRANCHE_LABELS).fillna("?")
display(tbl)

plot_tbl = tbl.sort_values("nombre", ascending=True)
fig_h = max(4.0, 0.35 * len(plot_tbl))
fig, ax = plt.subplots(figsize=(10, fig_h))
ax.barh(range(len(plot_tbl)), plot_tbl["nombre"], color="steelblue", edgecolor="white")
ax.set_yticks(range(len(plot_tbl)))
ax.set_yticklabels([f"{r.code} — {r.libelle}" for r in plot_tbl.itertuples()], fontsize=9)
ax.set_xlabel("Unités légales")
ax.set_title(f"Tranches effectif — {CFG['label'][:40]}")
plt.tight_layout()
plt.show()

In [ ]:
display(df["categorie_entreprise"].fillna("(n/a)").value_counts().to_frame("n"))
neo = pd.to_numeric(df["nombre_etablissements_ouverts"], errors="coerce")
mask_icp = (
    df["categorie_entreprise"].eq("PME")
    & df["tranche_effectif_salarie"].astype(str).isin(["11", "12"])
    & (neo >= 2)
)
print(f"Shortlist indicative (PME, 10-49 sal., >=2 établ.) : {mask_icp.sum():,}")